# JED Attack — Submission

EXFILTRATION-first verify-and-keep search (see CLAUDE.md). attack.py is the single source of truth; this notebook materializes it and serves the inference server.

In [ ]:
import sys, glob
from pathlib import Path
sys.argv = [sys.argv[0]]  # avoid argparse conflicts in notebooks
for c in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    root = str(Path(c).parent)
    if root not in sys.path:
        sys.path.insert(0, root)
    print('Dataset root:', root)
    break
print('Setup complete ✅')


In [ ]:
import base64
from pathlib import Path
_ATTACK_B64 = '...'
Path('/kaggle/working/attack.py').write_bytes(base64.b64decode(_ATTACK_B64))
print('Wrote /kaggle/working/attack.py ✅')


In [ ]:
import os, shutil
from pathlib import Path
WORKING_DIR = Path('/kaggle/working')
import kaggle_evaluation.jed_attack_134815.jed_attack_inference_server as server
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    server.JEDAttackInferenceServer().serve()
else:
    os.environ['AICOMP_MODEL_NAMES'] = 'deterministic'
    import kaggle_evaluation.jed_attack_134815.jed_attack_gateway as gw
    from aicomp_sdk.evaluation.ops import resolve_fixtures_dir
    gw.MODEL_NAMES = ['deterministic']
    server.DEFAULT_SUBMISSION_DIR = WORKING_DIR
    if not hasattr(gw.JEDAttackGateway, '_orig_run_attack'):
        gw.JEDAttackGateway._orig_run_attack = gw.JEDAttackGateway._run_attack_for_model
    _orig = gw.JEDAttackGateway._orig_run_attack
    def _limited(self, model_name, guardrail_factory=gw.OptimalGuardrail,
                 budget_s=gw.DEFAULT_BUDGET_S, seed=gw.ATTACK_SEED):
        return _orig(self, model_name, guardrail_factory=guardrail_factory,
                     budget_s=5.0, seed=seed)[:20]
    gw.JEDAttackGateway._run_attack_for_model = _limited
    try:
        server.JEDAttackInferenceServer().run_local_gateway(
            data_paths=(str(resolve_fixtures_dir()),))
    except Exception as _e:
        print('commit-stub local gateway note:', repr(_e))
    # FORMAT FIX (2026-06-26): the committed stub must carry the EXACT row Ids the
    # scored rerun produces (model x guardrail), or Kaggle rejects with 'Submission
    # Format Error'. The deterministic run yields 'deterministic_public' (wrong Id),
    # so OVERWRITE submission.csv with the canonical 4 rows (score 0; the rerun
    # replaces them with real scores). Independent of attack.py.
    _ROWS = ('gpt_oss_public', 'gpt_oss_private', 'gemma_public', 'gemma_private')
    with open(WORKING_DIR / 'submission.csv', 'w', newline='') as _f:
        _f.write('Id,Score\n')
        for _rid in _ROWS:
            _f.write(_rid + ',0.0\n')
    print('submission.csv produced:', (WORKING_DIR / 'submission.csv').exists())
    print(open(WORKING_DIR / 'submission.csv').read())
